In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn import set_config
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report,ConfusionMatrixDisplay, \
                            precision_score, recall_score, f1_score, roc_auc_score,roc_curve

In [4]:
df=pd.read_csv("Travel.csv")

In [185]:
df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


# Dataset cleaning

In [186]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CustomerID                4888 non-null   int64  
 1   ProdTaken                 4888 non-null   int64  
 2   Age                       4662 non-null   float64
 3   TypeofContact             4863 non-null   object 
 4   CityTier                  4888 non-null   int64  
 5   DurationOfPitch           4637 non-null   float64
 6   Occupation                4888 non-null   object 
 7   Gender                    4888 non-null   object 
 8   NumberOfPersonVisiting    4888 non-null   int64  
 9   NumberOfFollowups         4843 non-null   float64
 10  ProductPitched            4888 non-null   object 
 11  PreferredPropertyStar     4862 non-null   float64
 12  MaritalStatus             4888 non-null   object 
 13  NumberOfTrips             4748 non-null   float64
 14  Passport

In [187]:
df.describe()

,CustomerID,ProdTaken,Age,CityTier,DurationOfPitch,NumberOfPersonVisiting,NumberOfFollowups,PreferredPropertyStar,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,MonthlyIncome
count,4888.000000,4888.000000,4662.000000,4888.000000,4637.000000,4888.000000,4843.000000,4862.000000,4748.000000,4888.000000,4888.000000,4888.000000,4822.000000,4655.000000
mean,202443.500000,0.188216,37.622265,1.654255,15.490835,2.905074,3.708445,3.581037,3.236521,0.290917,3.078151,0.620295,1.187267,23619.853491
std,1411.188388,0.390925,9.316387,0.916583,8.519643,0.724891,1.002509,0.798009,1.849019,0.454232,1.365792,0.485363,0.857861,5380.698361
min,200000.000000,0.000000,18.000000,1.000000,5.000000,1.000000,1.000000,3.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1000.000000
25%,201221.750000,0.000000,31.000000,1.000000,9.000000,2.000000,3.000000,3.000000,2.000000,0.000000,2.000000,0.000000,1.000000,20346.000000
50%,202443.500000,0.000000,36.000000,1.000000,13.000000,3.000000,4.000000,3.000000,3.000000,0.000000,3.000000,1.000000,1.000000,22347.000000
75%,203665.250000,0.000000,44.000000,3.000000,20.000000,3.000000,4.000000,4.000000,4.000000,1.000000,4.000000,1.000000,2.000000,25571.000000
max,204887.000000,1.000000,61.000000,3.000000,127.000000,5.000000,6.000000,5.000000,22.000000,1.000000,5.000000,1.000000,3.000000,98678.000000


In [5]:
# dropping customerId as this is not required for prediction and has not impact on the target value
df=df.drop("CustomerID",axis=1)

In [60]:
X_train.shape, y_train.shape, X_test.shape,y_test.shape

((3421, 18), (3421,), (1467, 18), (1467,))

In [6]:
# Check for null values
print("Features null % in the dataset")
for features in df.columns:
    if(df[features].isnull().sum()>0):
        print(f"{features} : {df[features].isnull().sum()/len(df[features])*100}")

Features null % in the dataset
Age : 4.623567921440261
TypeofContact : 0.5114566284779051
DurationOfPitch : 5.1350245499181675
NumberOfFollowups : 0.9206219312602291
PreferredPropertyStar : 0.5319148936170213
NumberOfTrips : 2.8641571194762685
NumberOfChildrenVisiting : 1.3502454991816693
MonthlyIncome : 4.766775777414075


In [7]:
# Check the unique values from columns having null values
null_cols=[features for features in df.columns if(df[features].isnull().sum())>0]
for i in null_cols:
    print(f"Feature : {i} : \n Null count : {df[i].isnull().sum()} \n Unique values : {df[i].unique()}")

Feature : Age : 
 Null count : 226 
 Unique values : [41. 49. 37. 33. nan 32. 59. 30. 38. 36. 35. 31. 34. 28. 29. 22. 53. 21.
 42. 44. 46. 39. 24. 43. 50. 27. 26. 48. 55. 45. 56. 23. 51. 40. 54. 58.
 20. 25. 19. 57. 52. 47. 18. 60. 61.]
Feature : TypeofContact : 
 Null count : 25 
 Unique values : ['Self Enquiry' 'Company Invited' nan]
Feature : DurationOfPitch : 
 Null count : 251 
 Unique values : [  6.  14.   8.   9.  30.  29.  33.  22.  21.  32.  25.  27.  11.  17.
  15.  13.  12.  16.  10.  31.  18.  nan  24.  35.  28.  20.  26.  34.
  23.   5.  19. 126.   7.  36. 127.]
Feature : NumberOfFollowups : 
 Null count : 45 
 Unique values : [ 3.  4.  2.  5. nan  1.  6.]
Feature : PreferredPropertyStar : 
 Null count : 26 
 Unique values : [ 3.  4.  5. nan]
Feature : NumberOfTrips : 
 Null count : 140 
 Unique values : [ 1.  2.  7.  5.  6.  3.  4. 19. 21.  8. nan 20. 22.]
Feature : NumberOfChildrenVisiting : 
 Null count : 66 
 Unique values : [ 0.  2.  1. nan  3.]
Feature : MonthlyIncom

In [9]:
# Replacing null values
set_config(transform_output="pandas")
cols_to_impute=['TypeofContact','NumberOfFollowups','PreferredPropertyStar','DurationOfPitch']
imputecategorical=SimpleImputer(missing_values=np.nan,strategy="most_frequent",)
imputenumerical=SimpleImputer(missing_values=np.nan,strategy="mean")
imputenumzero=SimpleImputer(missing_values=np.nan,strategy="constant",fill_value=0)
coltransformer=ColumnTransformer([
    ("Categorical",imputecategorical,['TypeofContact','NumberOfFollowups','PreferredPropertyStar']),
    ("Numerical",imputenumerical,['DurationOfPitch','Age','NumberOfTrips']),
    ("Othernum",imputenumzero,['NumberOfChildrenVisiting'])],
                                 remainder='passthrough',
                                 verbose_feature_names_out=False
)
df=pd.DataFrame(coltransformer.fit_transform(df),columns=coltransformer.get_feature_names_out())

In [10]:
# Filling monthly missing values with grouped mean based on Designation
df['MonthlyIncome']=df.groupby("Designation")["MonthlyIncome"].transform(lambda x: x.fillna(x.median()))
df['MonthlyIncome'].isnull().sum()

np.int64(0)

In [11]:
# Checking the correlation between the columns
df['TotalPeopleVisiting']=df['NumberOfPersonVisiting']+df['NumberOfChildrenVisiting']

In [12]:
df.drop(['NumberOfChildrenVisiting','NumberOfPersonVisiting'],axis=1,inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   TypeofContact           4888 non-null   object 
 1   NumberOfFollowups       4888 non-null   object 
 2   PreferredPropertyStar   4888 non-null   object 
 3   DurationOfPitch         4888 non-null   float64
 4   Age                     4888 non-null   float64
 5   NumberOfTrips           4888 non-null   float64
 6   ProdTaken               4888 non-null   int64  
 7   CityTier                4888 non-null   int64  
 8   Occupation              4888 non-null   object 
 9   Gender                  4888 non-null   object 
 10  ProductPitched          4888 non-null   object 
 11  MaritalStatus           4888 non-null   object 
 12  Passport                4888 non-null   int64  
 13  PitchSatisfactionScore  4888 non-null   int64  
 14  OwnCar                  4888 non-null   

In [13]:
print("Features unique values for categorical columns in the dataset")
for features in df.select_dtypes(include=object).columns:
    print(f"{features} : {df[features].unique()}")

Features unique values for categorical columns in the dataset
TypeofContact : ['Self Enquiry' 'Company Invited']
NumberOfFollowups : [3.0 4.0 2.0 5.0 1.0 6.0]
PreferredPropertyStar : [3.0 4.0 5.0]
Occupation : ['Salaried' 'Free Lancer' 'Small Business' 'Large Business']
Gender : ['Female' 'Male' 'Fe Male']
ProductPitched : ['Deluxe' 'Basic' 'Standard' 'Super Deluxe' 'King']
MaritalStatus : ['Single' 'Divorced' 'Married' 'Unmarried']
Designation : ['Manager' 'Executive' 'Senior Manager' 'AVP' 'VP']


In [14]:
df.replace({
    'Gender':{'Fe Male':'Female'},
    'MaritalStatus':{'Unmarried':'Single'}
},inplace=True)

In [197]:
df.head()

,TypeofContact,NumberOfFollowups,PreferredPropertyStar,DurationOfPitch,Age,NumberOfTrips,ProdTaken,CityTier,Occupation,Gender,ProductPitched,MaritalStatus,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalPeopleVisiting
0,Self Enquiry,3.0,3.0,6.0,41.000000,1.0,1,3,Salaried,Female,Deluxe,Single,1,2,1,Manager,20993.0,3.0
1,Company Invited,4.0,4.0,14.0,49.000000,2.0,0,1,Salaried,Male,Deluxe,Divorced,0,3,1,Manager,20130.0,5.0
2,Self Enquiry,4.0,3.0,8.0,37.000000,7.0,1,1,Free Lancer,Male,Basic,Single,1,3,0,Executive,17090.0,3.0
3,Company Invited,3.0,3.0,9.0,33.000000,2.0,0,1,Salaried,Female,Basic,Divorced,1,5,1,Executive,17909.0,3.0
4,Self Enquiry,3.0,4.0,8.0,37.622265,1.0,0,1,Small Business,Male,Basic,Divorced,0,5,1,Executive,18468.0,2.0


In [15]:
# Creating independent and dependent features
X=df.drop("ProdTaken",axis=1)
y=df["ProdTaken"]
# Splitting the data in train and test
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

In [199]:
X_train.head()

,TypeofContact,NumberOfFollowups,PreferredPropertyStar,DurationOfPitch,Age,NumberOfTrips,CityTier,Occupation,Gender,ProductPitched,MaritalStatus,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalPeopleVisiting
736,Self Enquiry,4.0,3.0,10.0,48.0,1.0,1,Salaried,Male,Standard,Single,0,5,1,Senior Manager,25999.0,4.0
1615,Self Enquiry,3.0,5.0,11.0,30.0,6.0,1,Large Business,Female,Basic,Married,0,5,0,Executive,18204.0,3.0
336,Self Enquiry,5.0,5.0,14.0,29.0,2.0,1,Salaried,Male,Basic,Divorced,1,3,1,Executive,17119.0,4.0
4526,Self Enquiry,4.0,4.0,9.0,29.0,3.0,3,Small Business,Female,Deluxe,Married,1,3,1,Manager,23457.0,5.0
2665,Self Enquiry,5.0,4.0,11.0,34.0,8.0,1,Small Business,Female,Basic,Divorced,0,4,0,Executive,21300.0,4.0


In [200]:
y

0       1
1       0
2       1
3       0
4       0
       ..
4883    1
4884    1
4885    1
4886    1
4887    1
Name: ProdTaken, Length: 4888, dtype: int64

In [16]:
ohe=OneHotEncoder(handle_unknown='ignore',sparse_output=False,drop='first')
std_scalar=StandardScaler()
cat_features=X.select_dtypes(include='object').columns
num_features=X.select_dtypes(exclude='object').columns
ct=ColumnTransformer([
    ('OneHotEncoding',ohe,cat_features),
    ('standarscalar',std_scalar,num_features)
])


In [204]:
num_features

Index(['DurationOfPitch', 'Age', 'NumberOfTrips', 'ProdTaken', 'CityTier',
       'Passport', 'PitchSatisfactionScore', 'OwnCar', 'MonthlyIncome',
       'TotalPeopleVisiting'],
      dtype='object')

In [17]:
X_train=ct.fit_transform(X_train)

In [18]:
X_test=ct.transform(X_test)

# Training will multiple models and checking there accuracy

In [19]:
models={
    "Logistic":LogisticRegression(),
    "DecisionTree" : DecisionTreeClassifier(),
    "AdaBoost" : AdaBoostClassifier(),
    "RandomForest" : RandomForestClassifier(),
    "GradientBoost" : GradientBoostingClassifier()
}

for i in range(len(models.items())):
    model=list(models.values())[i]
    model.fit(X_train,y_train)
    y_pred_train=model.predict(X_train)
    y_pred_test=model.predict(X_test)

    print(f"Model :        : {list(models.keys())[i]}") 

    print("Train scores")
    print(f"Accuracy Score : {round(accuracy_score(y_train,y_pred_train),2)}")
    print(f"Precision Score: {round(precision_score(y_train,y_pred_train),2)}")
    print(f"Recall Score   : {round(recall_score(y_train,y_pred_train),2)}")
    print(f"ROC AUC Score   : {round(roc_auc_score(y_train,y_pred_train),2)}")
    print("-"*75)
    print(classification_report(y_train,y_pred_train))
    
    print("Test scores")
    print(f"Accuracy Score : {round(accuracy_score(y_test,y_pred_test),2)}")
    print(f"Precision Score: {round(precision_score(y_test,y_pred_test),2)}")
    print(f"Recall Score   : {round(recall_score(y_test,y_pred_test),2)}")
    print(f"ROC AUC Score   : {round(roc_auc_score(y_test,y_pred_test),2)}")
    print("-"*75)
    print(classification_report(y_test,y_pred_test))
    print("\n\n")

Model :        : Logistic
Train scores
Accuracy Score : 0.84
Precision Score: 0.69
Recall Score   : 0.31
ROC AUC Score   : 0.64
---------------------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.86      0.97      0.91      2775
           1       0.69      0.31      0.43       646

    accuracy                           0.84      3421
   macro avg       0.77      0.64      0.67      3421
weighted avg       0.83      0.84      0.82      3421

Test scores
Accuracy Score : 0.84
Precision Score: 0.65
Recall Score   : 0.32
ROC AUC Score   : 0.64
---------------------------------------------------------------------------
              precision    recall  f1-score   support

           0       0.86      0.96      0.91      1193
           1       0.65      0.32      0.43       274

    accuracy                           0.84      1467
   macro avg       0.75      0.64      0.67      1467
weighted avg       0

# Performing Hyperparameter tuning for all the models

In [27]:
rf_params={'n_estimators':[50,70],
           'criterion' : ['gini','entropy','log_loss'],
           'max_depth' : [2,5],
           'min_samples_split' : [2,4],
           'min_samples_leaf' : [1,3],
           'max_features' : ['sqrt','log2'],
           'class_weight' : ['balanced','balanced_subsample']
}

gb_params={'loss':['log_loss','exponential'],
           'learning_rate':[0.01,0.1],
           'n_estimators':[10,20,40],
           'criterion':['friedman_mse','squared_error']
          }

In [23]:
gsv=GridSearchCV(RandomForestClassifier(),param_grid=rf_params,scoring="accuracy",cv=5)
gsv.fit(X_train,y_train)
print(f"Best Params : {gsv.best_params_}")
print(f"Best Score : {gsv.best_score_}")

Best Params : {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}
Best Score : 0.8129205617449952


In [28]:
gsvgb=GridSearchCV(GradientBoostingClassifier(),param_grid=gb_params,scoring="accuracy",cv=5)
gsvgb.fit(X_train,y_train)
print(f"Best Params : {gsvgb.best_params_}")
print(f"Best Score : {gsvgb.best_score_}")

Best Params : {'criterion': 'friedman_mse', 'learning_rate': 0.1, 'loss': 'log_loss', 'n_estimators': 40}
Best Score : 0.8538447090963418


In [37]:
hypertuningcv={
    'RandomForest':[RandomForestClassifier(),rf_params],
    'GradientBoost':[GradientBoostingClassifier(),gb_params]
}
for i in range(len(list(hypertuningcv))):
    rsmodel=RandomizedSearchCV(list(hypertuningcv.values())[i][0],param_distributions=list(hypertuningcv.values())[i][1],scoring='accuracy',n_jobs=-1,cv=5)
    rsmodel.fit(X_train,y_train)
    print(f"Model : {(list(hypertuningcv))[i]} \n Best Params : {rsmodel.best_params_} \n Best Score : {rsmodel.best_score_}")

Model : RandomForest 
 Best Params : {'n_estimators': 70, 'min_samples_split': 2, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'max_depth': 5, 'criterion': 'gini', 'class_weight': 'balanced_subsample'} 
 Best Score : 0.805027532334486
Model : GradientBoost 
 Best Params : {'n_estimators': 40, 'loss': 'log_loss', 'learning_rate': 0.1, 'criterion': 'squared_error'} 
 Best Score : 0.8538447090963418
